# Notebook 4: OCR Benchmark
**ANPR System – MIPA UGM Parking Lot**

Benchmarks OCR performance on cropped license plate images:
- **CLA**: Character-Level Accuracy (Levenshtein-based)
- **PLA**: Plate-Level Accuracy (exact match ratio)
- Compares: EasyOCR vs Tesseract vs Fused (dual OCR)
- Tests with and without preprocessing (deskew, binarize, denoise, sharpen)
- Analyzes most common character confusion errors

## 1. Setup

In [ ]:
!pip install -q easyocr pytesseract python-Levenshtein
!apt-get install -q tesseract-ocr tesseract-ocr-eng

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ANPR_MIPA_UGM'

import os, cv2, re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from Levenshtein import distance as levenshtein_distance
import easyocr
import pytesseract

reader = easyocr.Reader(['en'], gpu=True)
print('Setup complete.')

## 2. Preprocessing Functions

In [ ]:
def apply_preprocessing(img):
    """Full OCR preprocessing: deskew, binarize, denoise, sharpen."""
    # Deskew
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 30, minLineLength=30, maxLineGap=10)
    if lines is not None:
        angles = [np.degrees(np.arctan2(y2-y1, x2-x1))
                  for line in lines for x1,y1,x2,y2 in line
                  if abs(np.degrees(np.arctan2(y2-y1, x2-x1))) <= 15]
        if angles:
            angle = np.median(angles)
            if abs(angle) >= 0.5:
                h, w = gray.shape
                M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
                gray = cv2.warpAffine(gray, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    # Binarize
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Denoise
    denoised = cv2.medianBlur(binary, 3)
    # Sharpen
    blurred = cv2.GaussianBlur(denoised, (0,0), 2.0)
    sharpened = cv2.addWeighted(denoised, 1.5, blurred, -0.5, 0)
    return sharpened

TESS_CONFIG = '--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

def run_easyocr(img):
    results = reader.readtext(img, detail=1)
    if not results:
        return '', 0.0
    text = ' '.join(r[1].upper().strip() for r in results)
    conf = np.mean([r[2] for r in results])
    return text, float(conf)

def run_tesseract(img):
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray, config=TESS_CONFIG).strip().upper()
    data = pytesseract.image_to_data(gray, config=TESS_CONFIG, output_type=pytesseract.Output.DICT)
    confs = [int(c)/100 for c, t in zip(data['conf'], data['text'])
             if int(c) > 0 and t.strip()]
    conf = np.mean(confs) if confs else 0.0
    return text, conf

def fuse_ocr(easy_text, easy_conf, tess_text, tess_conf, plate_regex=r'^[A-Z]{1,2}\s?\d{1,4}\s?[A-Z]{0,3}$'):
    easy_valid = bool(re.match(plate_regex, easy_text.replace(' ', '')))
    tess_valid = bool(re.match(plate_regex, tess_text.replace(' ', '')))
    if easy_valid and tess_valid:
        return (easy_text, easy_conf) if easy_conf >= tess_conf else (tess_text, tess_conf)
    if easy_valid:
        return easy_text, easy_conf
    if tess_valid:
        return tess_text, tess_conf
    return (easy_text, easy_conf) if easy_conf >= tess_conf else (tess_text, tess_conf)

def cla(pred, gt):
    p = pred.replace(' ', '').upper()
    g = gt.replace(' ', '').upper()
    if not g:
        return 1.0 if not p else 0.0
    return 1.0 - levenshtein_distance(p, g) / max(len(p), len(g))

def pla(pred, gt):
    return pred.replace(' ', '').upper() == gt.replace(' ', '').upper()

print('Functions defined.')

## 3. Load Test Data

In [ ]:
import csv
from pathlib import Path

# test_ocr.csv: columns image_path, ground_truth
TEST_CSV = f'{DRIVE_ROOT}/test_ocr.csv'

test_data = []
if Path(TEST_CSV).exists():
    with open(TEST_CSV, encoding='utf-8') as f:
        reader_csv = csv.DictReader(f)
        for row in reader_csv:
            test_data.append(row)
    print(f'Loaded {len(test_data)} test samples from CSV.')
else:
    # Fallback: scan for plate crops in a directory
    CROPS_DIR = f'{DRIVE_ROOT}/plate_crops'
    # Expects filenames like: AB1234CD.jpg (filename = ground truth)
    for img_path in Path(CROPS_DIR).glob('*.jpg'):
        gt = img_path.stem.upper()
        test_data.append({'image_path': str(img_path), 'ground_truth': gt})
    print(f'Loaded {len(test_data)} test samples from crops directory.')

print(f'Sample: {test_data[0] if test_data else "None"}')

## 4. Run OCR Benchmark

In [ ]:
results = []
for item in test_data:
    img = cv2.imread(item['image_path'])
    gt = item['ground_truth']
    if img is None:
        continue

    # Without preprocessing
    gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    easy_raw, easy_conf_raw = run_easyocr(gray_raw)
    tess_raw, tess_conf_raw = run_tesseract(gray_raw)
    fused_raw, fused_conf_raw = fuse_ocr(easy_raw, easy_conf_raw, tess_raw, tess_conf_raw)

    # With preprocessing
    processed = apply_preprocessing(img)
    easy_proc, easy_conf_proc = run_easyocr(processed)
    tess_proc, tess_conf_proc = run_tesseract(processed)
    fused_proc, fused_conf_proc = fuse_ocr(easy_proc, easy_conf_proc, tess_proc, tess_conf_proc)

    results.append({
        'ground_truth': gt,
        'easy_raw': easy_raw,  'easy_cla_raw': cla(easy_raw, gt), 'easy_pla_raw': pla(easy_raw, gt),
        'tess_raw': tess_raw,  'tess_cla_raw': cla(tess_raw, gt), 'tess_pla_raw': pla(tess_raw, gt),
        'fused_raw': fused_raw,'fused_cla_raw': cla(fused_raw, gt),'fused_pla_raw': pla(fused_raw, gt),
        'easy_proc': easy_proc,'easy_cla_proc': cla(easy_proc, gt),'easy_pla_proc': pla(easy_proc, gt),
        'tess_proc': tess_proc,'tess_cla_proc': cla(tess_proc, gt),'tess_pla_proc': pla(tess_proc, gt),
        'fused_proc': fused_proc,'fused_cla_proc': cla(fused_proc, gt),'fused_pla_proc': pla(fused_proc, gt),
    })

df_results = pd.DataFrame(results)
print(f'Evaluated {len(results)} samples.')

## 5. Summary Table

In [ ]:
conditions = [
    ('EasyOCR (no preprocess)',  'easy_cla_raw',  'easy_pla_raw'),
    ('Tesseract (no preprocess)','tess_cla_raw',  'tess_pla_raw'),
    ('Fused (no preprocess)',    'fused_cla_raw', 'fused_pla_raw'),
    ('EasyOCR (with preprocess)','easy_cla_proc', 'easy_pla_proc'),
    ('Tesseract (with preprocess)','tess_cla_proc','tess_pla_proc'),
    ('Fused (with preprocess)',  'fused_cla_proc','fused_pla_proc'),
]

summary_rows = []
for name, cla_col, pla_col in conditions:
    summary_rows.append({
        'Engine/Condition': name,
        'Avg CLA': df_results[cla_col].mean(),
        'PLA': df_results[pla_col].mean(),
        'n': len(df_results),
    })

df_summary = pd.DataFrame(summary_rows).round(4)
display(df_summary.style.highlight_max(subset=['Avg CLA','PLA'], color='lightgreen').format('{:.4f}', subset=['Avg CLA','PLA']))

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grouped bar: without vs with preprocessing for each engine
engines = ['EasyOCR', 'Tesseract', 'Fused']
cla_raw  = [df_results['easy_cla_raw'].mean(),  df_results['tess_cla_raw'].mean(),  df_results['fused_cla_raw'].mean()]
cla_proc = [df_results['easy_cla_proc'].mean(), df_results['tess_cla_proc'].mean(), df_results['fused_cla_proc'].mean()]
pla_raw  = [df_results['easy_pla_raw'].mean(),  df_results['tess_pla_raw'].mean(),  df_results['fused_pla_raw'].mean()]
pla_proc = [df_results['easy_pla_proc'].mean(), df_results['tess_pla_proc'].mean(), df_results['fused_pla_proc'].mean()]

x = np.arange(len(engines))
w = 0.35

# CLA plot
ax = axes[0]
b1 = ax.bar(x - w/2, cla_raw,  w, label='No Preprocessing', color='#4c72b0')
b2 = ax.bar(x + w/2, cla_proc, w, label='With Preprocessing', color='#55a868')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_title('Character-Level Accuracy (CLA)')
ax.set_xticks(x); ax.set_xticklabels(engines)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)

# PLA plot
ax = axes[1]
b3 = ax.bar(x - w/2, pla_raw,  w, label='No Preprocessing', color='#4c72b0')
b4 = ax.bar(x + w/2, pla_proc, w, label='With Preprocessing', color='#55a868')
for bars in [b3, b4]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_title('Plate-Level Accuracy (PLA)')
ax.set_xticks(x); ax.set_xticklabels(engines)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('OCR Engine Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/ocr_comparison.png', dpi=150)
plt.show()

## 7. Character Confusion Analysis

In [ ]:
from collections import Counter

# Analyze character-level errors for fused+preprocessed
errors = Counter()

for _, row in df_results.iterrows():
    pred = row['fused_proc'].replace(' ', '').upper()
    gt = row['ground_truth'].replace(' ', '').upper()
    # Simple aligned comparison (min length)
    for i, (p, g) in enumerate(zip(pred, gt)):
        if p != g:
            errors[(g, p)] += 1  # (correct, predicted_as)

print('Top 20 character confusion pairs (correct → predicted):')
for (correct, predicted), count in errors.most_common(20):
    print(f'  {correct} → {predicted}: {count} times')

# Heatmap
if errors:
    chars = sorted(set([c for pair in errors for c in pair]))
    matrix = np.zeros((len(chars), len(chars)), dtype=int)
    char_idx = {c: i for i, c in enumerate(chars)}
    for (correct, predicted), count in errors.items():
        if correct in char_idx and predicted in char_idx:
            matrix[char_idx[correct]][char_idx[predicted]] += count
    
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, xticklabels=chars, yticklabels=chars, annot=True, fmt='d',
                cmap='YlOrRd', ax=ax, linewidths=0.5)
    ax.set_xlabel('Predicted Character')
    ax.set_ylabel('Actual Character')
    ax.set_title('Character Confusion Matrix (Fused OCR + Preprocessing)')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/char_confusion.png', dpi=150)
    plt.show()